In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder,LabelEncoder
from sklearn.preprocessing import StandardScaler
import numpy as np
import re

#### 載入資料

In [2]:
df_train = pd.read_csv('train_data.csv')
df_test = pd.read_csv('test_day3.csv')
result_test_bid_id=df_test['bid_id'].copy()

In [3]:
df_train.columns

Index(['domain', 'ad_slot_format', 'ip', 'anonymous_url_id', 'key_page_url',
       'paying_price', 'click', 'city', 'ad_slot_width', 'user_tags',
       'creative_id', 'bidding_price', 'timestamp', 'bid_id',
       'ad_slot_visibility', 'ad_slot_height', 'url', 'region', 'ad_exchange',
       'user_agent', 'ad_slot_floor_price', 'ad_slot_id'],
      dtype='object')

In [4]:
df_train.shape

(1760309, 22)

In [4]:
df_test.columns

Index(['domain', 'ad_slot_format', 'ip', 'anonymous_url_id', 'key_page_url',
       'city', 'ad_slot_width', 'user_tags', 'creative_id', 'bidding_price',
       'timestamp', 'bid_id', 'ad_slot_visibility', 'ad_slot_height', 'url',
       'region', 'ad_exchange', 'user_agent', 'ad_slot_floor_price',
       'ad_slot_id'],
      dtype='object')

In [5]:
#sns.countplot(df_train['click'])
#plt.show()

#### one-hot編碼函式、labelencoding

In [6]:
def onehot_encode_column_fit(df, col_name):
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X = df[[col_name]]
    onehot = encoder.fit_transform(X)
    cols = encoder.get_feature_names_out([col_name])
    onehot_df = pd.DataFrame(onehot, columns=cols, index=df.index)
    print(f"[Train] 特徵 shape: {onehot_df.shape[1]}")
    df = df.drop(columns=[col_name]).join(onehot_df)
    return df, encoder, cols

def onehot_encode_column_transform(df, col_name, encoder, cols):
    X = df[[col_name]]
    onehot = encoder.transform(X)
    onehot_df = pd.DataFrame(onehot, columns=cols, index=df.index)
    print(f"[Test] 特徵 shape: {onehot_df.shape[1]}")
    df = df.drop(columns=[col_name]).join(onehot_df)
    return df
def onehot_encode_multiple_fit(df, col_names):
    encoders = {}
    all_cols = {}
    for col in col_names:
        df, encoder, cols = onehot_encode_column_fit(df, col)
        encoders[col] = encoder
        all_cols[col] = cols
    return df, encoders, all_cols

def onehot_encode_multiple_transform(df, col_names, encoders, all_cols):
    for col in col_names:
        df = onehot_encode_column_transform(df, col, encoders[col], all_cols[col])
    return df


#### ip處理

In [7]:
df_train['ip'].head(20)

0      115.45.195.*
1       120.40.95.*
2      60.163.144.*
3     123.120.244.*
4        222.75.4.*
5      113.200.77.*
6      110.190.28.*
7     110.179.226.*
8      123.139.96.*
9        60.12.97.*
10    121.225.158.*
11     125.40.181.*
12     183.27.100.*
13    183.190.105.*
14     14.151.252.*
15     125.39.238.*
16     123.11.222.*
17     122.195.76.*
18      122.79.66.*
19    222.216.140.*
Name: ip, dtype: object

In [8]:
def split_ip_column(df, ip_col='ip'):
    """
    將 IP 類似 '115.45.195.*' 切成三個欄位：ip_part1, ip_part2, ip_part3

    參數：
    - df: 輸入的 DataFrame
    - ip_col: 原始 IP 欄位名稱（預設為 'ip'）

    回傳：
    - 加上三個欄位後的 DataFrame
    """
    # 移除末尾的 '.*'
    df[ip_col + '_clean'] = df[ip_col].str.replace(r'\.\*$', '', regex=True)

    # 分割為三段
    ip_parts = df[ip_col + '_clean'].str.split('.', expand=True)
    df[ip_col + '_part1'] = ip_parts[0].astype(int)
    df[ip_col + '_part2'] = ip_parts[1].astype(int)
    df[ip_col + '_part3'] = ip_parts[2].astype(int)

    # 刪除中間處理欄位
    df.drop(columns=[ip_col + '_clean'], inplace=True)

    return df
# 對訓練集處理
df_train = split_ip_column(df_train, ip_col='ip')

# 對測試集處理（假設欄位名也是 'ip'）
df_test = split_ip_column(df_test, ip_col='ip')


#### ip_part2、3捨棄過於分散

In [9]:
top_45_ip_part1=df_train['ip_part1'].value_counts().head(45).index
# top 45 為原本的值，其餘為 999（或你自己指定的代碼）
df_train['ip_part1'] = df_train['ip_part1'].apply(lambda x: x if x in top_45_ip_part1 else 999)
df_test['ip_part1'] = df_test['ip_part1'].apply(lambda x: x if x in top_45_ip_part1 else 999)


#### ad_exchange、ad_slot_visibility、creative_id處理

#### domain處理

In [10]:
missing =df_train['domain'].isnull().sum() 
print(f'missing value count:{missing}')
df_train['domain'] = df_train['domain'].fillna('missing')
missing =df_train['domain'].isnull().sum()
print(f'missing value count:{missing}') 
# 測試集部分
missing =df_test['domain'].isnull().sum() 
print(f'missing value count:{missing}')
df_test['domain'] = df_test['domain'].fillna('missing')
missing =df_test['domain'].isnull().sum()
print(f'missing value count:{missing}') 

missing value count:94298
missing value count:0
missing value count:23879
missing value count:0


In [11]:
print(df_train.domain.value_counts().head(25))

domain
5F1RQS9rg5scFsf            196738
31xSTvprdN1RFt             176745
DFpETuxoGQdcFNKbuKz        135575
missing                     94298
trqRTuMvjTN7X9KbuKz         73473
3FKElpuEMusyJqKbuKz         67416
5F97t5E0BTK7XhNrUMpENpn     59725
ersbQv1RdoTy1m58uG          47954
DDTSQuf0MTTNaqKIvMpENpn     41511
31drTvprdN1RFt              38074
trqRTvFRLpscFU              34404
trqRTvFoMNmIFY5SaMpENpn     33081
trqRTJkrBoq7JsNr5SqfNX      29793
51F-l19YBqq4gsz4JKTI        20560
3FF-e59aG5syJqKbuKz         17938
trqRTv1xjQLI1m58uG          14668
trqRTJn7O95I1mKYUV          13374
trqRTJz7P9T4wTKbuKz         13205
Dr1NTvprdN1RFt              13041
trqRTvp8gIc7gspy            12434
kK1gq61NdoTy1m58uG          12194
5OC-q5uvgN171m58uG          12014
eKFjTu1Jgqc7gspy            11854
DFpETuFygZl7gspy            10498
DFpETJn8Penx1m54             9417
Name: count, dtype: int64


In [12]:
le = LabelEncoder()
top_25_domain=df_train['domain'].value_counts().head(25).index
df_train['domain_top25']=df_train['domain'].apply(lambda x: x if x in top_25_domain else 'other')
df_train['domain_labelencode']=le.fit_transform(df_train['domain_top25'])
#df_train = onehot_encode_column(df_train,'domain_labelencode')

# 測試集部分
df_test['domain_top25'] = df_test['domain'].apply(lambda x: x if x in top_25_domain else 'other')
df_test['domain_labelencode'] = le.transform(df_test['domain_top25']) 
#df_test = onehot_encode_column(df_test, 'domain_labelencode')

In [13]:
df_train['domain_labelencode'].value_counts()

domain_labelencode
17    570325
5     196738
1     176745
11    135575
16     94298
21     73473
3      67416
6      59725
14     47954
8      41511
0      38074
23     34404
24     33081
18     29793
4      20560
2      17938
22     14668
19     13374
20     13205
12     13041
25     12434
15     12194
7      12014
13     11854
10     10498
9       9417
Name: count, dtype: int64

In [14]:
df_test['domain_labelencode'].value_counts()

domain_labelencode
17    148446
1      41261
11     38231
5      33192
16     23879
21     22045
14     21466
3      20561
6      15061
2      12209
23     10586
24      9807
18      8745
4       7142
7       5727
9       4160
8       3963
25      3649
19      3375
20      2927
10      2762
0       2402
13      1970
15      1527
12       857
22       700
Name: count, dtype: int64

#### key_page_url、anonymous_url_id、url處理

In [15]:
missing =df_train['key_page_url'].isnull().sum() 
print(f'missing value count:{missing}')

missing value count:0


In [16]:
df_train.key_page_url.value_counts()

key_page_url
bebefa5efe83beee17a3d245e7c5085b    1760309
Name: count, dtype: int64

In [17]:
missing =df_train['anonymous_url_id'].isnull().sum() 
print(f'missing value count:{missing}')

missing value count:1760309


In [18]:
df_train.anonymous_url_id.value_counts()

Series([], Name: count, dtype: int64)

In [19]:
missing =df_train['url'].isnull().sum() 
print(f'url missing value count:{missing}')
df_train['url'] = df_train['url'].fillna('missing')
missing =df_train['url'].isnull().sum()
print(f'url missing value count:{missing}') 

url missing value count:36185
url missing value count:0


In [20]:
df_train.url.value_counts().head(10)

url
dedc488b98ca20707bc9a723957e7d1f    36675
missing                             36185
8dd142fa4bc566aa470b4c63e3da3d68    31828
bfa000ed663f4997db218d6da6b1510f    22562
47d63cb9aa310c4430e03126bb19adc1    19899
2a230d22d6725b124714a01f5ba6359     14618
823cbd7bc38ac495759ef5e5700a980e     8314
625d1b5916ea925332c7b326c0574cfa     6721
88dbab840de0cb9d623a68d07fd731b6     5998
64d74cca71f1c48cdb3f4a73d3c725db     5853
Name: count, dtype: int64

#### ad_slot_width、ad_slot_height、ad_slot_floor_price 處理

In [21]:
cols_to_scale = ['ad_slot_width', 'ad_slot_height', 'ad_slot_floor_price']
scaler = StandardScaler()
df_train[cols_to_scale] = scaler.fit_transform(df_train[cols_to_scale])
#測試集部分
df_test[cols_to_scale] = scaler.transform(df_test[cols_to_scale])

#### ad_slot_id 處理

In [22]:
missing=df_train['ad_slot_id'].isnull().sum()
print(f'missing:{missing}')

missing:0


In [23]:
df_train['ad_slot_id'].value_counts().head(30)

ad_slot_id
mm_10024662_3445902_11178345    132012
News_F_Width1                    90807
ALLINONE_F_Width1                89482
Ent_F_Width1                     80841
Ent_F_Upright                    57138
mm_10027070_2459574_9659312      52359
Ent_F_bottom_Width               38591
mm_10027070_118039_9659846       38273
mm_10027070_118039_10308280      36768
mm_10029307_121417_10790029      31758
News_F_Rectangle                 27736
Fashion_F_Rectangle              23193
mm_26632216_3300745_10762414     22757
Edu_F_Width1                     22329
3195670606                       21697
Fashion_F_Width1                 21096
mm_10024662_3445902_11178361     19742
mm_13604912_3392201_11480527     14669
mm_34022157_3445226_11175096     14163
mm_34022157_3445226_11175100     13793
News_Width5                      13186
Sports_F_Rectangle               12726
Astro_F_Upright                  12680
3798717662                       12670
News_F_bottom_Width              12015
mm_10024662_34

In [24]:
le = LabelEncoder()
top_30_adslotid=df_train['ad_slot_id'].value_counts().head(30).index
df_train['adslotid_top30']=df_train['ad_slot_id'].apply(lambda x: x if x in top_30_adslotid else 'other')
df_train['adslotid_labelencode']=le.fit_transform(df_train['adslotid_top30'])
#df_train = onehot_encode_column(df_train,'adslotid_labelencode')
# 測試集部分
df_test['adslotid_top30']=df_test['ad_slot_id'].apply(lambda x: x if x in top_30_adslotid else 'other')
df_test['adslotid_labelencode']=le.transform(df_test['adslotid_top30'])
#df_test = onehot_encode_column(df_test,'adslotid_labelencode')


#### user_tags 處理

In [25]:
def split_tags_to_cols(tags_str, max_tags=7):
    if not isinstance(tags_str, str) or tags_str == '':
        return [0] * max_tags
    tags = tags_str.split(',')
    # 取前 max_tags 個，長度不足補 0
    tags = tags[:max_tags] + ['0'] * (max_tags - len(tags))
    return tags

# 先把拆好的 list 變成 dataframe
tags_df = df_train['user_tags'].apply(split_tags_to_cols).apply(pd.Series)

# 把欄位命名成 user_tag_1, user_tag_2, ..., user_tag_7
tags_df.columns = [f'user_tag_{i+1}' for i in range(tags_df.shape[1])]
tags_df = tags_df.astype(int)  # 👈 加上這行就會變成 int

# 把新的欄位合併回原本 df
df_train = pd.concat([df_train, tags_df], axis=1)

#測試集部分

# 先把拆好的 list 變成 dataframe
tags_df = df_test['user_tags'].apply(split_tags_to_cols).apply(pd.Series)

# 把欄位命名成 user_tag_1, user_tag_2, ..., user_tag_7
tags_df.columns = [f'user_tag_{i+1}' for i in range(tags_df.shape[1])]
tags_df = tags_df.astype(int)  # 👈 加上這行就會變成 int

# 把新的欄位合併回原本 df
df_test = pd.concat([df_test, tags_df], axis=1)

In [26]:
print(f'user_tag1 type:{df_train["user_tag_1"].nunique()}')
print(f'user_tag2 type:{df_train["user_tag_2"].nunique()}')
print(f'user_tag3 type:{df_train["user_tag_3"].nunique()}')
print(f'user_tag4 type:{df_train["user_tag_4"].nunique()}')
print(f'user_tag5 type:{df_train["user_tag_5"].nunique()}')
print(f'user_tag6 type:{df_train["user_tag_6"].nunique()}')
print(f'user_tag7 type:{df_train["user_tag_7"].nunique()}')

user_tag1 type:44
user_tag2 type:44
user_tag3 type:44
user_tag4 type:44
user_tag5 type:44
user_tag6 type:44
user_tag7 type:44


#### time_stamp處理

In [27]:
df_train['day'] = df_train['timestamp'].str[8:10].astype(int)     # 第 9～10 位是日期
df_train['hour'] = df_train['timestamp'].str[11:13].astype(int)   # 第 12～13 位是小時

#測試集部分
df_test['day'] = df_test['timestamp'].str[8:10].astype(int)     # 第 9～10 位是日期
df_test['hour'] = df_test['timestamp'].str[11:13].astype(int)   # 第 12～13 位是小時

#### user_agent處理

In [28]:
# 假設 df['user_agent'] 已經有資料
def parse_user_agent(ua):
    if not isinstance(ua, str):
        ua = 'unknown'  
    ua = ua.lower()
    
    # 瀏覽器判斷
    if 'chrome' in ua:
        browser = 'Chrome'
        match = re.search(r'chrome/(\d+)', ua)
        version = int(match.group(1)) if match else -1
    elif 'msie' in ua or 'trident' in ua:
        browser = 'IE'
        match = re.search(r'msie (\d+)', ua)
        version = int(match.group(1)) if match else (11 if 'trident/7' in ua else -1)
    elif 'firefox' in ua:
        browser = 'Firefox'
        match = re.search(r'firefox/(\d+)', ua)
        version = int(match.group(1)) if match else -1
    elif 'safari' in ua and 'chrome' not in ua:
        browser = 'Safari'
        match = re.search(r'version/(\d+)', ua)
        version = int(match.group(1)) if match else -1
    else:
        browser = 'Other'
        version = -1
    
    # 作業系統判斷
    if 'windows nt 5.1' in ua:
        os = 'Windows XP'
    elif 'windows nt 6.1' in ua:
        os = 'Windows 7'
    elif 'android' in ua:
        os = 'Android'
    elif 'iphone' in ua or 'ipad' in ua or 'ios' in ua:
        os = 'iOS'
    elif 'windows nt 10' in ua:
        os = 'Windows 10'
    else:
        os = 'Other'

    # 是否為手機裝置
    is_mobile = int('android' in ua or 'iphone' in ua or 'ipad' in ua)

    # 是否為舊版 IE 或 Chrome < 30
    is_old_browser = int((browser == 'IE' and version <= 8) or (browser == 'Chrome' and version < 30))

    return pd.Series([browser, version, os, is_mobile, is_old_browser])

# 套用轉換
df_train[['browser', 'browser_version', 'os', 'is_mobile', 'is_old_browser']] = df_train['user_agent'].apply(parse_user_agent)

#df_train=onehot_encode_column(df_train,'browser')
#df_train=onehot_encode_column(df_train,'os')

#測試集部分
# 套用轉換
df_test[['browser', 'browser_version', 'os', 'is_mobile', 'is_old_browser']] = df_test['user_agent'].apply(parse_user_agent)

#df_test=onehot_encode_column(df_test,'browser')
#df_test=onehot_encode_column(df_test,'os')

#### city、region 處理

In [29]:
missing =df_train['city'].isnull().sum() 
print(f'city missing value count:{missing}')
missing =df_train['region'].isnull().sum() 
print(f'region missing value count:{missing}')

city missing value count:0
region missing value count:0


In [30]:
df_train['city'].value_counts().head(50)


city
1      82379
219    55199
217    49985
79     45749
275    42547
85     37063
277    36379
334    29328
2      28028
0      26373
81     23616
165    23162
184    23033
95     21862
4      20070
148    19618
82     17877
147    16519
233    15930
96     15558
56     14910
239    14047
202    13990
41     13873
97     13590
129    13158
66     13086
16     12746
222    12348
9      12227
107    12048
125    11673
309    11484
42     11447
84     11192
152    10910
126    10104
101    10099
153     9990
135     9973
375     9616
5       9498
159     8905
104     8879
83      8567
345     8523
8       8137
254     7880
154     7581
299     7523
Name: count, dtype: int64

In [31]:
df_train['region'].value_counts().head(30)

region
216    198057
80     141757
146    126594
276     98942
94      97712
3       88399
164     86522
1       82379
40      64308
183     62758
333     55928
15      54925
124     51486
106     51320
201     48266
79      45749
238     42820
275     42547
65      40232
55      39303
134     35156
2       28028
374     27229
308     26516
0       26373
27      25776
344     21094
298     19999
253      8534
368      7814
Name: count, dtype: int64

In [32]:
top_50_city = df_train['city'].value_counts().head(50).index
max_city_id = df_train['city'].max()
other_id = max_city_id + 1
df_train['city_top50'] = df_train['city'].apply(lambda x: x if x in top_50_city else other_id)

top_30_region=df_train['region'].value_counts().head(30).index
max_region_id=df_train['region'].max()
other_id=max_region_id+1
df_train['region_top30']=df_train['region'].apply(lambda x: x if x in top_30_region else other_id)

#測試集部分
#top_50_city = df_test['city'].value_counts().head(50).index
#max_city_id = df_test['city'].max()
#other_id = max_city_id + 1
df_test['city_top50'] = df_test['city'].apply(lambda x: x if x in top_50_city else other_id)

#top_30_region=df_test['region'].value_counts().head(30).index
#max_region_id=df_test['region'].max()
#other_id=max_region_id+1
df_test['region_top30']=df_test['region'].apply(lambda x: x if x in top_30_region else other_id)


In [33]:
df_train.columns

Index(['domain', 'ad_slot_format', 'ip', 'anonymous_url_id', 'key_page_url',
       'paying_price', 'click', 'city', 'ad_slot_width', 'user_tags',
       'creative_id', 'bidding_price', 'timestamp', 'bid_id',
       'ad_slot_visibility', 'ad_slot_height', 'url', 'region', 'ad_exchange',
       'user_agent', 'ad_slot_floor_price', 'ad_slot_id', 'ip_part1',
       'ip_part2', 'ip_part3', 'domain_top25', 'domain_labelencode',
       'adslotid_top30', 'adslotid_labelencode', 'user_tag_1', 'user_tag_2',
       'user_tag_3', 'user_tag_4', 'user_tag_5', 'user_tag_6', 'user_tag_7',
       'day', 'hour', 'browser', 'browser_version', 'os', 'is_mobile',
       'is_old_browser', 'city_top50', 'region_top30'],
      dtype='object')

#### 類別型特徵做one-hot encoding

In [34]:
categorical_columns = ['ad_exchange','ad_slot_visibility','creative_id','domain_labelencode',
                       'adslotid_labelencode','browser','os','city_top50','region_top30','ip_part1',
                       'user_tag_1','user_tag_2','user_tag_3','user_tag_4','user_tag_5','user_tag_6',
                       'user_tag_7']

# 對訓練集做 fit + transform
df_train, encoders, encoded_cols = onehot_encode_multiple_fit(df_train, categorical_columns)

# 對測試集只做 transform
df_test = onehot_encode_multiple_transform(df_test, categorical_columns, encoders, encoded_cols)

[Train] 特徵 shape: 3
[Train] 特徵 shape: 4
[Train] 特徵 shape: 8
[Train] 特徵 shape: 26
[Train] 特徵 shape: 31
[Train] 特徵 shape: 5
[Train] 特徵 shape: 5
[Train] 特徵 shape: 51
[Train] 特徵 shape: 31
[Train] 特徵 shape: 46
[Train] 特徵 shape: 44
[Train] 特徵 shape: 44
[Train] 特徵 shape: 44
[Train] 特徵 shape: 44
[Train] 特徵 shape: 44
[Train] 特徵 shape: 44
[Train] 特徵 shape: 44
[Test] 特徵 shape: 3
[Test] 特徵 shape: 4
[Test] 特徵 shape: 8
[Test] 特徵 shape: 26
[Test] 特徵 shape: 31
[Test] 特徵 shape: 5
[Test] 特徵 shape: 5
[Test] 特徵 shape: 51
[Test] 特徵 shape: 31
[Test] 特徵 shape: 46
[Test] 特徵 shape: 44
[Test] 特徵 shape: 44
[Test] 特徵 shape: 44
[Test] 特徵 shape: 44
[Test] 特徵 shape: 44
[Test] 特徵 shape: 44
[Test] 特徵 shape: 44


#### 除去沒有用到的特徵

In [35]:
df_train.columns

Index(['domain', 'ad_slot_format', 'ip', 'anonymous_url_id', 'key_page_url',
       'paying_price', 'click', 'city', 'ad_slot_width', 'user_tags',
       ...
       'user_tag_7_13678', 'user_tag_7_13776', 'user_tag_7_13800',
       'user_tag_7_13866', 'user_tag_7_13874', 'user_tag_7_14273',
       'user_tag_7_16593', 'user_tag_7_16617', 'user_tag_7_16661',
       'user_tag_7_16706'],
      dtype='object', length=546)

In [36]:
df_train=df_train.drop(columns=[
    'bidding_price','ip','url','key_page_url','city','paying_price'
    ,'bid_id','user_tags','timestamp','region','user_agent'
    ,'anonymous_url_id','domain','domain_top25','ad_slot_id','adslotid_top30',
    'ip_part2','ip_part3'])

In [37]:
df_train.columns

Index(['ad_slot_format', 'click', 'ad_slot_width', 'ad_slot_height',
       'ad_slot_floor_price', 'day', 'hour', 'browser_version', 'is_mobile',
       'is_old_browser',
       ...
       'user_tag_7_13678', 'user_tag_7_13776', 'user_tag_7_13800',
       'user_tag_7_13866', 'user_tag_7_13874', 'user_tag_7_14273',
       'user_tag_7_16593', 'user_tag_7_16617', 'user_tag_7_16661',
       'user_tag_7_16706'],
      dtype='object', length=528)

In [38]:
df_train.shape

(1760309, 528)

In [39]:
df_test.columns

Index(['domain', 'ad_slot_format', 'ip', 'anonymous_url_id', 'key_page_url',
       'city', 'ad_slot_width', 'user_tags', 'bidding_price', 'timestamp',
       ...
       'user_tag_7_13678', 'user_tag_7_13776', 'user_tag_7_13800',
       'user_tag_7_13866', 'user_tag_7_13874', 'user_tag_7_14273',
       'user_tag_7_16593', 'user_tag_7_16617', 'user_tag_7_16661',
       'user_tag_7_16706'],
      dtype='object', length=544)

In [40]:
df_test=df_test.drop(columns=[
    'ip','url','key_page_url','city','bidding_price'
    ,'user_tags','timestamp','region','user_agent'
    ,'ad_slot_id','anonymous_url_id','domain','domain_top25','adslotid_top30',
    'ip_part2','ip_part3'])

In [41]:
df_test.shape

(446650, 528)

### 導出資料

In [42]:
df_train[df_train.select_dtypes(include='float64').columns] = df_train.select_dtypes(include='float64').astype('float32')
df_test[df_test.select_dtypes(include='float64').columns] = df_test.select_dtypes(include='float64').astype('float32')
df_train[df_train.select_dtypes(include='int64').columns] = df_train.select_dtypes(include='int64').astype('int32')
df_test[df_test.select_dtypes(include='int64').columns] = df_test.select_dtypes(include='int64').astype('int32')


In [43]:
df_train.to_csv('train_data_v2.csv')
df_test.to_csv('test_day3_v2.csv')

In [44]:
print(df_train.dtypes.value_counts())

float32    521
int32        6
bool         1
Name: count, dtype: int64
